In [1]:
import textwrap

import numpy as np
import pandas as pd
import xarray as xr

from matplotlib import pyplot as plt

In [2]:
ds = xr.open_mfdataset("data/*.nc", chunks="auto")

In [3]:
# compute the wind speed
ws = np.sqrt(ds.u10**2 + ds.v10**2)
ws.attrs["long_name"] = "10 metre wind speed"
ws.attrs["units"] = ds.u10.units

In [4]:
# compute the relative humidity
d2m_degC = ds.d2m - 273.15
t2m_degC = ds.t2m - 273.15

beta = 17.625
lamda = 243.04

r = 100.0 * np.exp((beta * d2m_degC) / (lamda + d2m_degC)) / np.exp((beta * t2m_degC) / (lamda + t2m_degC))
r.attrs["long_name"] = "Relative humidity"
r.attrs["units"] = "%"

In [5]:
ds2 = ds.drop_vars(["u10", "v10", "d2m"]).assign(ws=ws, r=r)
ds2

<xarray.Dataset> Size: 46GB
Dimensions:     (number: 10, valid_time: 488, latitude: 361, longitude: 720)
Coordinates:
  * number      (number) int64 80B 0 1 2 3 4 5 6 7 8 9
  * valid_time  (valid_time) datetime64[ns] 4kB 2023-06-01 ... 2023-12-31T21:...
  * latitude    (latitude) float64 3kB 90.0 89.5 89.0 88.5 ... -89.0 -89.5 -90.0
  * longitude   (longitude) float64 6kB 0.0 0.5 1.0 1.5 ... 358.5 359.0 359.5
    expver      (valid_time) <U4 8kB dask.array<chunksize=(488,), meta=np.ndarray>
Data variables:
    ssr         (number, valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(2, 196, 146, 288), meta=np.ndarray>
    cp          (number, valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(2, 196, 146, 288), meta=np.ndarray>
    lsp         (number, valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(2, 196, 146, 288), meta=np.ndarray>
    t2m         (number, valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(2, 196, 146, 288), meta=np.ndarray>
    sp          (number, valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(2, 196, 146, 288), meta=np.ndarray>
    tcc         (number, valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(2, 196, 146, 288), meta=np.ndarray>
    tco3        (number, valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(2, 196, 146, 288), meta=np.ndarray>
    ws          (number, valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(2, 196, 146, 288), meta=np.ndarray>
    r           (number, valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(2, 196, 146, 288), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2024-12-18T09:54 GRIB to CDM+CF via cfgrib-0.9.1...

In [6]:
def plot_ensemble_mean_spread(ds, key, ax, sample=slice(None)):
    var = ds[key]
    
    mean = var.mean(dim="number")
    spread = var.std(dim="number")

    min_abs = float(spread.where(spread > 0.0).min(skipna=True))

    abs_mean = np.abs(mean)
    min_rel = spread.where(abs_mean > 0.0) / abs_mean.where(abs_mean > 0.0)
    min_rel = float(min_rel.where(min_rel > 0.0).min(skipna=True))

    months = sorted(set(zip(
        pd.to_datetime(var.valid_time).year,
        pd.to_datetime(var.valid_time).month,
    )))
    
    ax.set_title(f"{'\n'.join(textwrap.wrap(var.long_name, width=22, break_long_words=False))}")
    ax.set_xlabel(f"mean [{var.units}]")
    ax.set_ylabel(f"spread [{var.units}]")

    if key in ("ssr", "cp", "lsp"):
        ax.set_xscale("log")
        ax.set_yscale("log")

    handles = []

    for year, month in months:
        handles.append(ax.scatter(
            mean.sel(valid_time=f"{year}-{month:02}").values.flatten()[sample],
            spread.sel(valid_time=f"{year}-{month:02}").values.flatten()[sample],
            s=1, label=f"{year}/{month:02}", edgecolor="none", marker=".",
        ))

    legends = []

    legends.append(ax.axhline(min_abs, c="red", ls=":", label=f"abs = {min_abs:.03}"))

    xlim, ylim = ax.get_xlim(), ax.get_ylim()

    rlim = sorted([xlim[0], 0.0, xlim[1]])

    legends.extend(ax.plot(rlim, np.array(rlim) * min_rel * np.sign(rlim), c="blue", ls=":", label=f"rel = {min_rel:.03}"))
    legends.extend(ax.plot(rlim, np.array(rlim) * 0.01 * np.sign(rlim), c="blue", ls="--", label="rel = 1%"))
    legends.extend(ax.plot(rlim, np.array(rlim) * 0.05 * np.sign(rlim), c="blue", ls="-.", label="rel = 5%"))

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    
    ax.legend(handles=legends)

    return handles

In [ ]:
fig, axs = plt.subplots(3, 3, figsize=(9, 9))
fig.suptitle("ERA5 Ensemble mean vs spread")

for key, ax in zip(ds2, axs.flatten()):
    handles = plot_ensemble_mean_spread(ds2, key, ax, sample=slice(0, None, 5000))
    break

fig.legend(handles=handles, loc="upper right", ncols=len(handles), markerscale=10)

fig.tight_layout()
plt.show()